In [1]:
import xarray as xr
import dask.array as da
from dask.distributed import LocalCluster, Client, performance_report
import numpy as np
import psutil
import os
from dask.diagnostics import ProgressBar

# Auto-configure based on available cores
total_cores = psutil.cpu_count(logical=True)
n_workers = max(1, total_cores // 2)  # Conservative split
threads_per_worker = max(1, total_cores // n_workers)

cluster = LocalCluster(
    n_workers=2,               # Use only 2 worker processes
    threads_per_worker=2,      # Each with 2 threads
    worker_dashboard_address=False,
    diagnostics_port=None
)

client = Client(cluster)


client = Client(cluster)
client


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 2
Total threads: 4,Total memory: 98.23 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:45327,Workers: 2
Dashboard: http://127.0.0.1:8787/status,Total threads: 4
Started: Just now,Total memory: 98.23 GiB
Comm: tcp://127.0.0.1:41275,Total threads: 2
Dashboard: http://127.0.0.1:42879/status,Memory: 49.12 GiB
Nanny: tcp://127.0.0.1:35671,


In [2]:
from openeo.local import LocalConnection

# Initialize the local connection
local_conn = LocalConnection("./")

# Define the STAC collection URL
stac_item = "https://stac.intertwin.fedcloud.eu/collections/ERA5_T2M_SSRD_TP"

# Specify the temporal extent
temporal_extent = ["2000-01-01", "2020-12-31"]

# Specify the spatial extent (bounding box)
spatial_extent = {
    "west": 4,
    "east": 16,
    "south": 42,
    "north": 51
}

# Load the data cube with specified parameters
era5_single = local_conn.load_stac(
    url=stac_item,
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
).execute().to_dataset(dim='bands')

# Display the dataset
stac_item = "https://stac.intertwin.fedcloud.eu/collections/ERA5_PRESSURE"

from openeo.local import LocalConnection
local_conn = LocalConnection("./")

era5_pressure = local_conn.load_stac(
    url=stac_item,
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
).execute().to_dataset(dim='bands')
 
ERA5 = xr.merge([era5_single, era5_pressure])
ERA5

<xarray.Dataset> Size: 445MB
Dimensions:  (time: 7670, lat: 37, lon: 49)
Coordinates:
  * lat      (lat) float64 296B 51.0 50.75 50.5 50.25 ... 42.75 42.5 42.25 42.0
  * lon      (lon) float64 392B 4.0 4.25 4.5 4.75 5.0 ... 15.25 15.5 15.75 16.0
  * time     (time) datetime64[ns] 61kB 2000-01-01 2000-01-02 ... 2020-12-30
Data variables:
    ssrd     (time, lat, lon) float32 56MB dask.array<chunksize=(500, 37, 49), meta=np.ndarray>
    t2m      (time, lat, lon) float32 56MB dask.array<chunksize=(500, 37, 49), meta=np.ndarray>
    tp       (time, lat, lon) float32 56MB dask.array<chunksize=(500, 37, 49), meta=np.ndarray>
    q_850    (time, lat, lon) float32 56MB dask.array<chunksize=(500, 37, 49), meta=np.ndarray>
    t_850    (time, lat, lon) float32 56MB dask.array<chunksize=(500, 37, 49), meta=np.ndarray>
    u_850    (time, lat, lon) float32 56MB dask.array<chunksize=(500, 37, 49), meta=np.ndarray>
    v_850    (time, lat, lon) float32 56MB dask.array<chunksize=(500, 37, 49), meta=np.ndarray>
    z_850    (time, lat, lon) float32 56MB dask.array<chunksize=(500, 37, 49), meta=np.ndarray>
Attributes:
    crs:      EPSG:4326

In [3]:
stac_item = "https://stac.intertwin.fedcloud.eu/collections/EMO1_DEM"

from openeo.local import LocalConnection
local_conn = LocalConnection("./")

dem = local_conn.load_stac(
    url=stac_item,
    spatial_extent=spatial_extent,
    bands=["dem"]
).execute()
dem = dem.to_dataset(dim='bands')["dem"].to_dataset()
dem = dem.isel(time=0)
dem = dem.drop_vars("time")
dem

<xarray.Dataset> Size: 2MB
Dimensions:  (lat: 540, lon: 720)
Coordinates:
  * lat      (lat) float64 4kB 50.99 50.97 50.96 50.94 ... 42.04 42.02 42.01
  * lon      (lon) float64 6kB 4.008 4.025 4.042 4.058 ... 15.96 15.98 15.99
Data variables:
    dem      (lat, lon) float32 2MB dask.array<chunksize=(121, 421), meta=np.ndarray>

## REMAPPING

In [4]:
import xarray as xr
import numpy as np

def match_to_mid_resolution(source_ds, target_ds, lat_name='lat', lon_name='lon',
                            num_mid_lats=None, num_mid_lons=None):
    """
    Interpolates both source and target datasets to a common mid-resolution grid,
    then expands the target (e.g., DEM) to match source's non-spatial dimensions and aligns chunks.
    
    Parameters:
        source_ds: xarray.Dataset, multi-dimensional (e.g. time, lat, lon)
        target_ds: xarray.Dataset, typically with lat/lon only (e.g. DEM)
        lat_name: str, name of latitude coordinate
        lon_name: str, name of longitude coordinate
        num_mid_lats: int, number of mid-resolution latitudes (default: half of target's)
        num_mid_lons: int, number of mid-resolution longitudes (default: half of target's)

    Returns:
        Tuple of (interpolated source_ds, expanded/interpolated target_ds) with aligned chunks
    """
    # Default mid-resolution size
    if num_mid_lats is None:
        num_mid_lats = len(target_ds[lat_name])
    if num_mid_lons is None:
        num_mid_lons = len(target_ds[lon_name])

    # Find common lat/lon bounds
    min_lat = max(source_ds[lat_name].min().item(), target_ds[lat_name].min().item())
    max_lat = min(source_ds[lat_name].max().item(), target_ds[lat_name].max().item())
    min_lon = max(source_ds[lon_name].min().item(), target_ds[lon_name].min().item())
    max_lon = min(source_ds[lon_name].max().item(), target_ds[lon_name].max().item())

    # Mid-resolution coordinate arrays
    mid_lats = np.linspace(min_lat, max_lat, num_mid_lats)
    mid_lons = np.linspace(min_lon, max_lon, num_mid_lons)

    mid_coords = {
        lat_name: xr.DataArray(mid_lats, dims=lat_name),
        lon_name: xr.DataArray(mid_lons, dims=lon_name)
    }

    # Interpolate both datasets
    source_mid = source_ds.interp(mid_coords, method='linear')
    target_mid = target_ds.interp(mid_coords, method='linear')

    # Expand target to match source's additional dimensions (e.g., time, ensemble)
    extra_dims = {dim: source_mid.coords[dim] for dim in source_mid.dims
                  if dim not in [lat_name, lon_name]}

    if extra_dims:
        target_expanded = target_mid.expand_dims(extra_dims).broadcast_like(source_mid)
    else:
        target_expanded = target_mid

    # Align chunks if either is dask-backed
    source_mid, target_expanded = xr.align(source_mid, target_expanded)

    target_expanded = target_expanded.chunk(source_mid.chunks)

    returned = xr.merge([source_mid, target_expanded]).astype('float32')

    return returned

ERA5_interpolated = match_to_mid_resolution(ERA5, dem)
ERA5_interpolated

<xarray.Dataset> Size: 107GB
Dimensions:  (time: 7670, lat: 540, lon: 720)
Coordinates:
  * time     (time) datetime64[ns] 61kB 2000-01-01 2000-01-02 ... 2020-12-30
  * lat      (lat) float64 4kB 42.01 42.02 42.04 42.06 ... 50.96 50.98 50.99
  * lon      (lon) float64 6kB 4.008 4.025 4.042 4.058 ... 15.96 15.97 15.99
Data variables:
    ssrd     (time, lat, lon) float32 12GB dask.array<chunksize=(500, 540, 720), meta=np.ndarray>
    t2m      (time, lat, lon) float32 12GB dask.array<chunksize=(500, 540, 720), meta=np.ndarray>
    tp       (time, lat, lon) float32 12GB dask.array<chunksize=(500, 540, 720), meta=np.ndarray>
    q_850    (time, lat, lon) float32 12GB dask.array<chunksize=(500, 540, 720), meta=np.ndarray>
    t_850    (time, lat, lon) float32 12GB dask.array<chunksize=(500, 540, 720), meta=np.ndarray>
    u_850    (time, lat, lon) float32 12GB dask.array<chunksize=(500, 540, 720), meta=np.ndarray>
    v_850    (time, lat, lon) float32 12GB dask.array<chunksize=(500, 540, 720), meta=np.ndarray>
    z_850    (time, lat, lon) float32 12GB dask.array<chunksize=(500, 540, 720), meta=np.ndarray>
    dem      (time, lat, lon) float32 12GB dask.array<chunksize=(500, 540, 720), meta=np.ndarray>
Attributes:
    crs:      EPSG:4326

Cyclic Feature Addition

In [5]:
import numpy as np
import dask.array as da
from datetime import date


def encode_cyclical_features(values, max_value):
    """Encode cyclical features using sine and cosine transformations."""
    sin = np.sin(2 * np.pi * values / max_value)
    cos = np.cos(2 * np.pi * values / max_value)
    return sin, cos

def repeat_along_axis(arr, repeats, axis):
    """Repeat array along specified axis."""
    return da.repeat(arr[None, ...], repeats, axis=axis)

def get_spatial_dims(ds):
    """
    Detect spatial dimension names in the dataset.
    Returns (y_dim, x_dim) tuple based on common naming conventions.
    """
    dims = set(ds.dims)
    
    y_candidates = ['y', 'lat', 'latitude', 'lats']
    x_candidates = ['x', 'lon', 'longitude', 'long', 'lons']
    
    y_dim = next((d for d in y_candidates if d in dims), None)
    x_dim = next((d for d in x_candidates if d in dims), None)
    
    if y_dim is None or x_dim is None:
        raise ValueError(
            f"Could not detect spatial dimensions. Available dimensions: {list(dims)}. "
            f"Tried y names: {y_candidates}, x names: {x_candidates}"
        )
    
    return y_dim, x_dim

def get_existing_chunks(ds, dims):
    """
    Get chunking pattern from existing variables in the dataset.
    Returns dict of {dim: chunksize} for the specified dimensions.
    """
    chunks = {}
    for var in ds.data_vars.values():
        if hasattr(var.data, 'chunks'):
            var_chunks = dict(zip(var.dims, var.data.chunks))
            for dim in dims:
                if dim in var_chunks and dim not in chunks:
                    # Take first chunk size found for each dimension
                    chunks[dim] = var_chunks[dim][0]
        if all(dim in chunks for dim in dims):
            break
    return chunks or None

def encode_doys(ds, time_dim='time', spatial_dims=None, extra_dims=None, inplace=False):
    """
    Encode day of year as cyclical features and add to dataset, handling:
    - ERA5 (time, y, x)
    - SEAS5 (time, number, y, x) or similar
    
    Parameters:
    -----------
    ds : xarray.Dataset
        Input dataset containing time dimension
    time_dim : str, optional
        Name of time dimension (default: 'time')
    spatial_dims : tuple, optional
        Explicit (y_dim, x_dim) names. If None, auto-detect.
    extra_dims : list, optional
        Additional dimensions (e.g., ['number']) to repeat along
    inplace : bool, optional
        If True, modify dataset in place (default: False)
    
    Returns:
    --------
    xarray.Dataset
        Dataset with sin_doy and cos_doy variables added
    """
    if not inplace:
        ds = ds.copy()
    
    # Auto-detect spatial dims if not provided
    if spatial_dims is None:
        y_dim, x_dim = get_spatial_dims(ds)
    else:
        y_dim, x_dim = spatial_dims
    
    # Determine all target dimensions
    target_dims = [time_dim]
    if extra_dims:
        target_dims.extend(extra_dims)
    target_dims.extend([y_dim, x_dim])
    
    # Get chunking pattern from existing variables
    chunks = get_existing_chunks(ds, target_dims)
    
    # Compute day of year
    doys = ds[time_dim].values.astype('datetime64[D]')
    doys = da.asarray([date.timetuple(doy.astype(object)).tm_yday for doy in doys])
    
    # Encode cyclical features
    sin_doy, cos_doy = encode_cyclical_features(doys, 365)
    
    # Reshape and repeat along all non-time dimensions
    for dim in target_dims[1:]:  # Skip time_dim
        repeats = len(ds[dim])
        sin_doy = da.repeat(sin_doy[..., None], repeats, axis=-1)
        cos_doy = da.repeat(cos_doy[..., None], repeats, axis=-1)
    
    # Reshape to final dimensions
    sin_doy = sin_doy.reshape([len(ds[dim]) for dim in target_dims])
    cos_doy = cos_doy.reshape([len(ds[dim]) for dim in target_dims])
    
    # Apply chunking if found
    if chunks:
        chunk_sizes = [chunks.get(dim, -1) for dim in target_dims]
        sin_doy = sin_doy.rechunk(chunk_sizes)
        cos_doy = cos_doy.rechunk(chunk_sizes)
    
    # Add to dataset
    ds['sin_doy'] = (target_dims, sin_doy)
    ds['cos_doy'] = (target_dims, cos_doy)
    
    # Add attributes
    for name in ['sin_doy', 'cos_doy']:
        ds[name].attrs.update({
            'long_name': f"{'Sine' if 'sin' in name else 'Cosine'} of day of year",
            'units': 'unitless',
            'description': f"Cyclical encoding of day of year"
        })
    
    return ds

    # For ERA5 (only time, y, x)
encode_doys(ERA5_interpolated, inplace=True)



<xarray.Dataset> Size: 155GB
Dimensions:  (time: 7670, lat: 540, lon: 720)
Coordinates:
  * time     (time) datetime64[ns] 61kB 2000-01-01 2000-01-02 ... 2020-12-30
  * lat      (lat) float64 4kB 42.01 42.02 42.04 42.06 ... 50.96 50.98 50.99
  * lon      (lon) float64 6kB 4.008 4.025 4.042 4.058 ... 15.96 15.97 15.99
Data variables:
    ssrd     (time, lat, lon) float32 12GB dask.array<chunksize=(500, 540, 720), meta=np.ndarray>
    t2m      (time, lat, lon) float32 12GB dask.array<chunksize=(500, 540, 720), meta=np.ndarray>
    tp       (time, lat, lon) float32 12GB dask.array<chunksize=(500, 540, 720), meta=np.ndarray>
    q_850    (time, lat, lon) float32 12GB dask.array<chunksize=(500, 540, 720), meta=np.ndarray>
    t_850    (time, lat, lon) float32 12GB dask.array<chunksize=(500, 540, 720), meta=np.ndarray>
    u_850    (time, lat, lon) float32 12GB dask.array<chunksize=(500, 540, 720), meta=np.ndarray>
    v_850    (time, lat, lon) float32 12GB dask.array<chunksize=(500, 540, 720), meta=np.ndarray>
    z_850    (time, lat, lon) float32 12GB dask.array<chunksize=(500, 540, 720), meta=np.ndarray>
    dem      (time, lat, lon) float32 12GB dask.array<chunksize=(500, 540, 720), meta=np.ndarray>
    sin_doy  (time, lat, lon) float64 24GB dask.array<chunksize=(500, 540, 720), meta=np.ndarray>
    cos_doy  (time, lat, lon) float64 24GB dask.array<chunksize=(500, 540, 720), meta=np.ndarray>
Attributes:
    crs:      EPSG:4326

In [8]:

# Path to your Zarr store
zarr_path = "/mnt/CEPH_PROJECTS/InterTwin/Climate_Downscaling/EMO1_DOWNSCALING/stage_1/ERA5_to_latent_by_3.zarr"

# Loop through each variable and save it one by one
for i, var_name in enumerate(ERA5_chunked.data_vars):
    print(f"Processing variable {i+1}/{len(ERA5_chunked.data_vars)}: {var_name}")
    
    # Select only the current variable
    var_ds = ERA5_chunked[[var_name]]
    
    # Write to Zarr (use 'w' mode for the first variable, 'a' for subsequent ones)
    mode = "w" if i == 0 else "a"
    var_ds.to_zarr(zarr_path, mode=mode, compute=True)

print("All variables saved successfully!")

Processing variable 1/11: ssrd
Processing variable 2/11: t2m
Processing variable 3/11: tp
Processing variable 4/11: q_850
Processing variable 5/11: t_850
Processing variable 6/11: u_850
Processing variable 7/11: v_850
Processing variable 8/11: z_850
Processing variable 9/11: dem
Processing variable 10/11: sin_doy
Processing variable 11/11: cos_doy
All variables saved successfully!


In [1]:
import xarray as xr

ds = xr.open_zarr("/mnt/CEPH_PROJECTS/InterTwin/Climate_Downscaling/EMO1_DOWNSCALING/stage_1/ERA5_to_latent_by_3.zarr")
ds_sel = ds.sel(time=slice("2015-01-01", "2020-12-31"))

<xarray.Dataset> Size: 17GB
Dimensions:  (time: 7670, lat: 180, lon: 240)
Coordinates:
  * lat      (lat) float64 1kB 42.01 42.06 42.11 42.16 ... 50.89 50.94 50.99
  * lon      (lon) float64 2kB 4.008 4.058 4.109 4.159 ... 15.89 15.94 15.99
  * time     (time) datetime64[ns] 61kB 2000-01-01 2000-01-02 ... 2020-12-30
Data variables:
    cos_doy  (time, lat, lon) float64 3GB dask.array<chunksize=(1000, 50, 50), meta=np.ndarray>
    dem      (time, lat, lon) float32 1GB dask.array<chunksize=(1000, 50, 50), meta=np.ndarray>
    q_850    (time, lat, lon) float32 1GB dask.array<chunksize=(1000, 50, 50), meta=np.ndarray>
    sin_doy  (time, lat, lon) float64 3GB dask.array<chunksize=(1000, 50, 50), meta=np.ndarray>
    ssrd     (time, lat, lon) float32 1GB dask.array<chunksize=(1000, 50, 50), meta=np.ndarray>
    t2m      (time, lat, lon) float32 1GB dask.array<chunksize=(1000, 50, 50), meta=np.ndarray>
    t_850    (time, lat, lon) float32 1GB dask.array<chunksize=(1000, 50, 50), meta=np.ndarray>
    tp       (time, lat, lon) float32 1GB dask.array<chunksize=(1000, 50, 50), meta=np.ndarray>
    u_850    (time, lat, lon) float32 1GB dask.array<chunksize=(1000, 50, 50), meta=np.ndarray>
    v_850    (time, lat, lon) float32 1GB dask.array<chunksize=(1000, 50, 50), meta=np.ndarray>
    z_850    (time, lat, lon) float32 1GB dask.array<chunksize=(1000, 50, 50), meta=np.ndarray>
Attributes:
    crs:      EPSG:4326

In [1]:
import xarray as xr

ds = xr.open_zarr("/mnt/CEPH_PROJECTS/InterTwin/Climate_Downscaling/EMO1_DOWNSCALING/stage_1/v3/SEAS5/SEAS5_to_latent_AUGUST_2021.zarr")
ds

<xarray.Dataset> Size: 35GB
Dimensions:  (time: 216, y: 240, x: 360, number: 51)
Coordinates:
  * number   (number) int32 204B 0 1 2 3 4 5 6 7 8 ... 43 44 45 46 47 48 49 50
  * time     (time) datetime64[ns] 2kB 2021-08-01 2021-08-02 ... 2022-03-04
  * x        (x) float64 3kB 4.0 4.033 4.067 4.1 4.134 ... 15.9 15.93 15.97 16.0
  * y        (y) float64 2kB 42.0 42.04 42.08 42.11 ... 50.89 50.92 50.96 51.0
Data variables:
    cos_doy  (time, y, x) float64 149MB dask.array<chunksize=(216, 180, 240), meta=np.ndarray>
    dem      (time, number, y, x) float32 4GB dask.array<chunksize=(216, 1, 180, 240), meta=np.ndarray>
    q_850    (time, number, y, x) float32 4GB dask.array<chunksize=(216, 1, 180, 240), meta=np.ndarray>
    sin_doy  (time, y, x) float64 149MB dask.array<chunksize=(216, 180, 240), meta=np.ndarray>
    ssrd     (time, number, y, x) float32 4GB dask.array<chunksize=(216, 1, 180, 240), meta=np.ndarray>
    t2m      (time, number, y, x) float32 4GB dask.array<chunksize=(216, 1, 180, 240), meta=np.ndarray>
    t_850    (time, number, y, x) float32 4GB dask.array<chunksize=(216, 1, 180, 240), meta=np.ndarray>
    tp       (time, number, y, x) float32 4GB dask.array<chunksize=(216, 1, 180, 240), meta=np.ndarray>
    u_850    (time, number, y, x) float32 4GB dask.array<chunksize=(216, 1, 180, 240), meta=np.ndarray>
    v_850    (time, number, y, x) float32 4GB dask.array<chunksize=(216, 1, 180, 240), meta=np.ndarray>
    z_850    (time, number, y, x) float32 4GB dask.array<chunksize=(216, 1, 180, 240), meta=np.ndarray>
Attributes:
    crs:      EPSG:4326

In [9]:
import xarray as xr

ds = xr.open_zarr("/mnt/CEPH_PROJECTS/InterTwin/Climate_Downscaling/EMO1_DOWNSCALING/stage_1/v3/SEAS5/SEAS5_to_latent_APRIL_2022.zarr/")
ds

<xarray.Dataset> Size: 17GB
Dimensions:  (time: 216, y: 180, x: 240, number: 51)
Coordinates:
  * number   (number) int32 204B 0 1 2 3 4 5 6 7 8 ... 43 44 45 46 47 48 49 50
  * time     (time) datetime64[ns] 2kB 2022-04-01 2022-04-02 ... 2022-11-02
  * x        (x) float64 2kB 4.008 4.058 4.109 4.159 ... 15.84 15.89 15.94 15.99
  * y        (y) float64 1kB 42.01 42.06 42.11 42.16 ... 50.84 50.89 50.94 50.99
Data variables:
    cos_doy  (time, y, x) float64 75MB dask.array<chunksize=(216, 180, 240), meta=np.ndarray>
    dem      (time, number, y, x) float32 2GB dask.array<chunksize=(216, 1, 180, 240), meta=np.ndarray>
    q_850    (time, number, y, x) float32 2GB dask.array<chunksize=(216, 1, 180, 240), meta=np.ndarray>
    sin_doy  (time, y, x) float64 75MB dask.array<chunksize=(216, 180, 240), meta=np.ndarray>
    ssrd     (time, number, y, x) float32 2GB dask.array<chunksize=(216, 1, 180, 240), meta=np.ndarray>
    t2m      (time, number, y, x) float32 2GB dask.array<chunksize=(216, 1, 180, 240), meta=np.ndarray>
    t_850    (time, number, y, x) float32 2GB dask.array<chunksize=(216, 1, 180, 240), meta=np.ndarray>
    tp       (time, number, y, x) float32 2GB dask.array<chunksize=(216, 1, 180, 240), meta=np.ndarray>
    u_850    (time, number, y, x) float32 2GB dask.array<chunksize=(216, 1, 180, 240), meta=np.ndarray>
    v_850    (time, number, y, x) float32 2GB dask.array<chunksize=(216, 1, 180, 240), meta=np.ndarray>
    z_850    (time, number, y, x) float32 2GB dask.array<chunksize=(216, 1, 180, 240), meta=np.ndarray>
Attributes:
    crs:      EPSG:4326

In [10]:
ds["cos_doy"] = ds["cos_doy"].expand_dims("number").broadcast_like(ds["dem"])
ds["sin_doy"] = ds["sin_doy"].expand_dims("number").broadcast_like(ds["dem"])

ds

ValueError: cannot reindex or align along dimension 'number' because of conflicting dimension sizes: {1, 51} (note: an index is found along that dimension with size=51)